# CXR-DetectBench Phase 2 - Data Preprocessing

## 1. Environment Setup

In [ ]:
!pip install -q --no-cache-dir ensemble-boxes pycocotools opencv-python-headless ultralytics scikit-learn tqdm matplotlib

In [ ]:
# GPU compatibility check: P100 (sm_60) needs PyTorch <= 2.4
# Kaggle default PyTorch 2.10 dropped sm_60 support
import subprocess, sys

def check_gpu():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            encoding='utf-8')
        return out.strip()
    except:
        return 'unknown'

gpu_name = check_gpu()
print(f'GPU: {gpu_name}')

CU121_URL = 'https://download.pytorch.org/whl/cu121'

if 'P100' in gpu_name:
    print('P100 detected! Installing PyTorch 2.4 (supports sm_60)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'torch==2.4.0', 'torchvision==0.19.0',
        '--index-url', CU121_URL])
    print('PyTorch 2.4 installed. Restart kernel may be needed.')
elif 'T4' in gpu_name:
    print('T4 detected! Default PyTorch is compatible.')
else:
    print(f'GPU={gpu_name}, proceeding with default PyTorch...')

In [ ]:
# Verify PyTorch CUDA
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    x = torch.zeros(1).cuda()
    print('GPU test passed!')
else:
    print('WARNING: CUDA not available')

## 2. Explore Inputs

In [ ]:
import os, glob
from pathlib import Path
os.chdir('/kaggle/working')
print('Available inputs:')
!ls -la /kaggle/input/

## 3. Clone Repository

In [ ]:
!git clone https://github.com/kinjazA/cxr-detectbench.git
%cd cxr-detectbench

## 4. Prepare Data

In [ ]:
!mkdir -p data/raw data/processed

from pathlib import Path
import os
import shutil

# Find and copy train.csv
found_csv = False
for p in Path('/kaggle/input').rglob('train.csv'):
    print(f'Found train.csv: {p}')
    !cp '{p}' data/raw/train.csv
    found_csv = True
    break
if not found_csv:
    print('Downloading train.csv from Kaggle...')
    !kaggle competitions download vinbigdata-chest-xray-abnormalities-detection -f train.csv -p data/raw/
    !unzip -o data/raw/train.csv.zip -d data/raw/ 2>/dev/null || true

# Find and copy train_meta.csv/images.csv (image dimensions)
found_meta = False
for name in ['train_meta.csv', 'images.csv', 'image_meta.csv']:
    for p in Path('/kaggle/input').rglob(name):
        print(f'Found {name}: {p}')
        !cp '{p}' data/raw/images.csv
        found_meta = True
        break
    if found_meta:
        break
if not found_meta:
    raise FileNotFoundError('No train_meta.csv/images.csv found under /kaggle/input')

# Link PNG images. The target must directly contain <image_id>.png files.
png_candidates = []
for meta_path in Path('/kaggle/input').rglob('train_meta.csv'):
    train_png_dir = meta_path.parent / 'train'
    png_count = len(list(train_png_dir.glob('*.png'))) if train_png_dir.is_dir() else 0
    print(f'Candidate from train_meta: {train_png_dir} ({png_count} PNGs)')
    if png_count > 0:
        png_candidates.append((train_png_dir, png_count))

if len(png_candidates) != 1:
    raise RuntimeError(f'Expected exactly one non-empty train PNG dir from train_meta.csv, got: {png_candidates}')

train_png_dir, png_count = png_candidates[0]
images_link = Path('data/processed/images_png')
if images_link.is_symlink() or images_link.is_file():
    images_link.unlink()
elif images_link.exists():
    shutil.rmtree(images_link)
images_link.symlink_to(train_png_dir, target_is_directory=True)

linked_count = len(list(images_link.glob('*.png')))
print(f'Linked PNG dir: {images_link} -> {os.path.realpath(images_link)}')
print(f'Linked {linked_count} PNG images')
if linked_count == 0:
    raise RuntimeError(f'Linked image directory has 0 PNGs: {images_link} -> {os.path.realpath(images_link)}')

print('=== data/raw/ ===')
!ls -lh data/raw/
print('=== data/processed/ ===')
!ls -lh data/processed/


## 5. Multi-Annotator Fusion

In [ ]:
!python scripts/label_fusion.py \
    --train_csv data/raw/train.csv \
    --images_csv data/raw/images.csv \
    --output_dir data/processed/labels_coco \
    --iou_thr 0.5

In [ ]:
!python scripts/check_fusion_output.py

## 6. Fusion Ablation

In [ ]:
!python scripts/run_fusion_ablation.py \
    --coco_dir data/processed/labels_coco \
    --images_dir data/processed/images_png \
    --train_csv data/raw/train.csv \
    --epochs 20 \
    --imgsz 640 \
    --batch 16 \
    --output phase2_fusion_ablation.csv

In [ ]:
import pandas as pd
r = Path('phase2_fusion_ablation.csv')
if r.exists():
    df = pd.read_csv(r)
    print('='*60)
    print('Phase 2.4 Fusion Ablation Results')
    print('='*60)
    print(df.to_string(index=False))
    v = df[df['mAP50'] > 0]
    if len(v):
        b = v.loc[v['mAP50'].idxmax(), 'fusion_mode']
        print(f'Best fusion: {b}')
else:
    print('No results yet')

## 7. Final YOLO Labels

In [ ]:
BEST = 'wbf'
!python scripts/convert_coco_yolo.py \
    --coco_json data/processed/labels_coco/{BEST}/annotations.json \
    --output_dir data/processed/labels_yolo/{BEST}

## 8. Verify

In [ ]:
print('=== Phase 2 Complete ===')
print('COCO:')
!ls -lh data/processed/labels_coco/*/annotations.json
print('YOLO:')
!ls data/processed/labels_yolo/{BEST}/ | head -10
print('Ablation:')
!cat phase2_fusion_ablation.csv 2>/dev/null || echo 'Not found'
print('Remember to Save Version!')